In [29]:
import ssl

ssl._create_default_https_context = ssl._create_unverified_context

In [1]:
import os

In [3]:
%pwd

'c:\\Users\\xuanl\\ChickenDisease-hx\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'c:\\Users\\xuanl\\ChickenDisease-hx'

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [7]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [8]:
class ConfigurationManager: 
    def __init__(
            self, 
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH
    ): 
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig: 
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir = config.root_dir,
            source_URL = config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir,
        )

        return data_ingestion_config

In [9]:
import os
import urllib.request as request
import zipfile
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size

In [10]:
class DataIngestion: 
    def __init__(self, config: DataIngestionConfig): 
        self.config = config

    def download_file(self): 
        if not os.path.exists(self.config.local_data_file): 
            filename, headers = request.urlretrieve(
                url = self.config.source_URL, 
                filename = self.config.local_data_file
            )
            logger.info(f"File downloaded at: {filename} with following headers: {headers}")
        else: 
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")
        
    
    def extract_zip_file(self): 
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [30]:
import traceback

try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()

    data_ingestion = DataIngestion(config=data_ingestion_config)

    data_ingestion.download_file()
    data_ingestion.extract_zip_file()

except Exception as e:
    print("ERROR:", e)
    traceback.print_exc()

[2026-06-18 02:18:42,303: INFO: common] yaml file: config\config.yaml loaded successfully
[2026-06-18 02:18:42,306: INFO: common] yaml file: params.yaml loaded successfully
[2026-06-18 02:18:42,308: INFO: common] directory created at: artifacts
[2026-06-18 02:18:42,310: INFO: common] directory created at: artifacts/data_ingestion
[2026-06-18 02:18:44,858: INFO: 708910430] File downloaded at: artifacts/data_ingestion/data.zip with following headers: Connection: close
Content-Length: 11616915
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "adf745abc03891fe493c3be264ec012691fe3fa21d861f35a27edbe6d86a76b1"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: ADFE:288D17:15A9E2:319666:6A32E501
Accept-Ranges: bytes
Date: Wed, 17 Jun 2026 18:18:44 GMT
Via: 1.1 varnish
X-Served-By: cache-kul9826-KUL
X-C

In [21]:
import ssl
print(ssl.OPENSSL_VERSION)

OpenSSL 3.5.7 9 Jun 2026


In [22]:
import certifi
print(certifi.where())

c:\Users\xuanl\Downloads\Anaconda\envs\MUAI-class\lib\site-packages\certifi\cacert.pem
